# 03 — Pipeline STT

Deepgram, Whisper, AssemblyAI, Soniox, Speechmatics, Cartesia.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
%load_ext autoreload
%autoreload 2

import _setup
env = _setup.load()
print(f'getpatter version target: {env.patter_version}')


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


In [ ]:
import sys
import getpatter
with _setup.cell('version_check', tier=1, env=env) as ok:
    if ok:
        print(f'getpatter {getpatter.__version__} on Python {sys.version.split()[0]}')
        assert getpatter.__version__ >= env.patter_version, \
            f'installed {getpatter.__version__} < target {env.patter_version}'


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


In [ ]:
from getpatter import Patter, Twilio
with _setup.cell('local_mode', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(
                account_sid='ACtest00000000000000000000000000',
                auth_token='test',
            ),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        # Verify carrier is wired up via LocalConfig.
        assert p._local_config.telephony_provider == 'twilio'
        assert p._local_config.phone_number == '+15555550100'
        print(f'provider={p._local_config.telephony_provider}  phone={p._local_config.phone_number}')


### Cloud mode (coming soon)
When `api_key=` is supported, Patter cloud handles telephony. For now, the SDK raises `NotImplementedError` — this cell verifies the guard.


In [ ]:
from getpatter import Patter
with _setup.cell('cloud_mode', tier=1, env=env) as ok:
    if ok:
        try:
            Patter(api_key='pt_test_xxx')
            raise AssertionError('expected NotImplementedError — cloud mode guard missing')
        except NotImplementedError as exc:
            print(f'cloud mode guard OK: {exc}')


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the engine from `engine=` / `stt=`/`tts=`.


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI
with _setup.cell('agent_engines', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000',
                           auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        rt = p.agent(system_prompt='hi', engine=OpenAIRealtime(api_key='sk-test'))
        cv = p.agent(system_prompt='hi', engine=ElevenLabsConvAI(api_key='el-test', agent_id='a1'))
        pl = p.agent(system_prompt='hi')  # default: OpenAI Realtime fallback
        assert rt.provider == 'openai_realtime', rt.provider
        assert cv.provider == 'elevenlabs_convai', cv.provider
        assert pl.provider == 'openai_realtime', pl.provider
        print(f'rt.provider={rt.provider}  cv.provider={cv.provider}  pl.provider={pl.provider}')


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


### Cloud mode
Same SDK, just an `api_key=` instead of a carrier — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the mode from `engine=` / `stt=`/`tts=`.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises STT provider construction, audio transcoding, and (T3) live transcription.


### STT provider construction


In [ ]:
from getpatter import DeepgramSTT, WhisperSTT, OpenAITranscribeSTT
with _setup.cell('stt_providers', tier=1, env=env) as ok:
    if ok:
        dg = DeepgramSTT(api_key='dg-test', model='nova-2', language='en-US')
        wh = WhisperSTT(api_key='sk-test', model='whisper-1')
        ot = OpenAITranscribeSTT(api_key='sk-test')
        print(f'Deepgram: model={dg.model}  lang={dg.language}')
        print(f'Whisper:  model={wh.model}')
        print(f'OpenAI Transcribe: provider={ot.provider if hasattr(ot, "provider") else type(ot).__name__}')
        assert dg.model == 'nova-2'


### μ-law ↔ PCM-16 transcoding roundtrip


In [ ]:
from getpatter import mulaw_to_pcm16, pcm16_to_mulaw
with _setup.cell('mulaw_transcoding', tier=1, env=env) as ok:
    if ok:
        # 100ms of silence at 8kHz = 800 bytes
        import struct
        SAMPLES = 800
        pcm_original = struct.pack('<' + 'h' * SAMPLES, *([0] * SAMPLES))
        mulaw_bytes = pcm16_to_mulaw(pcm_original)
        pcm_recovered = mulaw_to_pcm16(mulaw_bytes)
        print(f'PCM original:  {len(pcm_original)} bytes ({SAMPLES} samples)')
        print(f'μ-law encoded: {len(mulaw_bytes)} bytes (8-bit, 2:1 compression)')
        print(f'PCM recovered: {len(pcm_recovered)} bytes')
        assert len(mulaw_bytes) == SAMPLES
        assert len(pcm_recovered) == len(pcm_original)


### 8kHz → 16kHz resampling


In [ ]:
from getpatter import resample_8k_to_16k, resample_16k_to_8k
with _setup.cell('resampler', tier=1, env=env) as ok:
    if ok:
        import struct
        # 100ms silence at 8kHz
        pcm_8k = struct.pack('<' + 'h' * 800, *([0] * 800))
        pcm_16k = resample_8k_to_16k(pcm_8k)
        pcm_8k_back = resample_16k_to_8k(pcm_16k)
        print(f'8kHz input:   {len(pcm_8k)} bytes ({len(pcm_8k)//2} samples)')
        print(f'16kHz output: {len(pcm_16k)} bytes ({len(pcm_16k)//2} samples)')
        print(f'8kHz round-trip: {len(pcm_8k_back)} bytes')
        assert len(pcm_16k) == len(pcm_8k) * 2


### Live: Deepgram transcription  *(T3 — requires `DEEPGRAM_API_KEY`)*

Transcribes a synthetic fixture WAV using the Deepgram REST API.


In [ ]:
import httpx
with _setup.cell('deepgram_live', tier=3, required=['deepgram_key'], env=env) as ok:
    if ok:
        audio = _setup.load_fixture('audio/hello_world_16khz_pcm.wav')
        resp = httpx.post(
            'https://api.deepgram.com/v1/listen?model=nova-2&language=en-US',
            headers={
                'Authorization': f'Token {env.deepgram_key}',
                'Content-Type': 'audio/wav',
            },
            content=audio,
            timeout=30,
        )
        resp.raise_for_status()
        transcript = resp.json()['results']['channels'][0]['alternatives'][0]['transcript']
        print(f'Deepgram transcript: {transcript!r}')


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Calls a real number through the Pipeline engine using Deepgram STT. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
with _setup.cell('live_preflight', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'DEEPGRAM_API_KEY'], env=env) as ok:
    if ok:
        print(f'  carrier:  Twilio {env.twilio_number}  →  {env.target_number}')
        print(f'  STT:      Deepgram  (nova-2-general)')
        print(f'  webhook:  {env.public_webhook_url or "(ngrok auto-launch)"}')


### Live Pipeline STT call *(T4)*


In [ ]:
from getpatter import Patter, Twilio, DeepgramSTT, OpenAILLM, OpenAITTS
with _setup.cell('live_stt_call', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'DEEPGRAM_API_KEY', 'OPENAI_API_KEY'], env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid=env.twilio_sid, auth_token=env.twilio_token),
            phone_number=env.twilio_number,
            webhook_url=env.public_webhook_url,
        )
        agent = p.agent(
            system_prompt='Greet the caller and say goodbye.',
            stt=DeepgramSTT(api_key=env.deepgram_key),
            llm=OpenAILLM(api_key=env.openai_key, model='gpt-4o-mini'),
            tts=OpenAITTS(api_key=env.openai_key, voice='alloy'),
        )
        try:
            await p.call(env.target_number, agent=agent, first_message='Hello from Patter STT demo.',
                         ring_timeout=env.max_call_seconds)
            print('✓ Pipeline STT call completed')
        finally:
            _setup.hangup_leftover_calls(env)
